# pandas 之十二：数据结构转换（宽表 ↔ 长表）

---

前面处理的数据基本是"一张表一个长相"。但真实表格数据有两种常见长相，必须能自由互转：

- **宽表（wide）**：列多、行少，**一列装一个指标** —— 人看着舒服，Excel 里最常见
- **长表（long / tidy）**：行多、列少，用**"一列装指标名、一列装数值"**来描述 —— 机器算着舒服

`seaborn` 画图、`pivot_table` 做统计、很多建模库都要求**长表**；
而你要交给别人看的时候，又往往得变回**宽表**。

> 一句话：**宽表给人看，长表给机器算。**

## 学习目标

1. 分清宽表 / 长表，知道什么时候要转
2. **一列拆成多列**：`str.split(expand=True)`、`apply(pd.Series)`
3. **宽变长**：`stack()`、`melt()`
4. **长变宽**：`unstack()`、`pivot()`、`pivot_table()`
5. DataFrame ↔ 字典：`to_dict()` 的 7 种 `orient`


In [ ]:
import pandas as pd

pd.set_option('display.unicode.east_asian_width', True)   # 中文对齐好看一点
print('pandas 版本：', pd.__version__)


---
# 一、先看清「宽表」和「长表」

同一个班的三门课成绩，用两种方式摆出来：


In [ ]:
# 【宽表】一列一个科目
wide = pd.DataFrame({
    '数学': [90, 85, 95],
    '语文': [88, 92, 80],
    '英语': [78, 90, 85],
}, index=['小明', '小红', '小刚'])

print('【宽表】宽 = 列数，一列一个指标：')
print(wide)
print('形状：', wide.shape, '（3 行，3 列）')


In [ ]:
# 【长表】用两列描述"哪个指标、值是多少"
long = pd.DataFrame({
    '姓名': ['小明', '小明', '小明', '小红', '小红', '小红'],
    '科目': ['数学', '语文', '英语', '数学', '语文', '英语'],
    '成绩': [90, 88, 78, 85, 92, 90],
})

print('【长表】一行一个观测值：')
print(long)
print('形状：', long.shape, '（6 行，3 列）')


两种摆法的**信息量完全一样**，只是用法不同：

| 维度 | 宽表 | 长表 |
|---|---|---|
| 长相 | 列多行少 | 行多列少 |
| 适合 | 给人看、导出 Excel | 给机器算、画图、建模 |
| 增加指标 | 要**加列** | 只是**加行** |
| pandas 里怎么来 | `pivot` / `pivot_table` / `unstack` | `melt` / `stack` |

> **判断标准**：如果新增一个"类别"（比如再加一门「物理」），
> 宽表要**改表结构**（加一列），长表只要**加几行**。
> 后者才是数据库和数据分析里"干净"的形态。


---
# 二、一列拆成多列

开发中经常遇到这种"塞在一起的脏数据"：一列里装了用逗号分隔的多个值。
我们要把它炸开成多列。


## 2.1 字符串拆分：`str.split(expand=True)`

`str.split()` 加上 `expand=True` 这个参数，返回的就是 **DataFrame**（而不是一列列表）。


In [ ]:
data = pd.DataFrame({'a': ['a,b,c', 'd,e,f', 'g,h,i']})
print('拆分前：')
print(data)

# 【关键】expand=True 才会展开成多列；不加这个参数会得到一列"列表"
print('\n不加 expand（得到一列列表，不好用）：')
print(data['a'].str.split(',').to_frame())

print('\nexpand=True（直接得到 3 列）：')
split_df = data['a'].str.split(',', expand=True)
print(split_df)


In [ ]:
# 给它起个像样的列名，再拼回原表
data[['部分1', '部分2', '部分3']] = data['a'].str.split(',', expand=True)
data = data.drop(columns=['a'])          # 原来那列不需要了
print('拆完并命名：')
print(data)


## 2.2 列数不确定怎么办？—— 用 `join` 自动拼接

上面每行都是 3 段，所以能写死 `['部分1','部分2','部分3']`。
但如果有的行是 3 段、有的是 4 段呢？


In [ ]:
data2 = pd.DataFrame({'a': ['a,b,c', 'd,e,f', 'g,h,i,j']})   # 注意第 3 行有 4 段
print('原始数据（各行段数不一样）：')
print(data2)

split_wide = data2['a'].str.split(',', expand=True)
print('\nexpand=True 会自动按"最长的行"对齐，缺的位置补 NaN：')
print(split_wide)

print('\n给列名加前缀（add_prefix），再 join 回原表：')
result2 = data2.join(split_wide.add_prefix('段'))
print(result2)


> **小技巧**：`add_prefix('段')` 能把列名批量变成 `段0, 段1, 段2...`，
> 比手动写列名安全得多 —— **列数会变的时候就该这么干**。


## 2.3 列里装的是列表 / 元组？用 `apply(pd.Series)`

如果单元格里存的**不是字符串**，而是真正的 `list` / `tuple`，
那么 `str.split` 就用不上了（它只处理字符串），要用 `apply` + `pd.Series`。


In [ ]:
data3 = pd.DataFrame({'a': [('甲', '乙', '丙'), ('丁', '戊', '己'), ('庚', '辛', '壬')]})
print('列里装的是元组：')
print(data3)

# 【关键】apply 会把每个元组交给 pd.Series，自动按位置对齐成多列
split3 = data3['a'].apply(lambda x: pd.Series(x))
print('\n拆成 3 列（列名是默认的 0/1/2）：')
print(split3)


In [ ]:
split3.columns = ['第一', '第二', '第三']
print('改名后拼回原表：')
print(pd.concat([data3, split3], axis=1))


> **和 `explode` 的区别**（第十一篇讲过 `explode`）：
>
> ![explode 与 分列 的区别](image.png)
>
> | 方法 | 做什么 | 结果 | 应用场景 |
> |---|---|---|---|
> | `str.split(expand=True)` / `apply(pd.Series)` | 把**一列拆成多列** | 列变多，行数不变 | 一格里是多个**不同类**的值（姓名+年龄） |
> | `explode()` | 把**一列列表炸成多行** | 行变多，列数不变 | 一格里是多个**同类**的值（多个标签） |
>
> 方向正好相反，别搞混。
> **判断诀窍**：拆出来的东西是"**新的不同字段**"（如 省/市/区）→ 分列；
> 拆出来的是"**同一种东西的多份**"（如 标签A/标签B/标签C）→ `explode`。


---
# 三、宽变长（列转行）

现在把第一节那张宽表，变成长表。有两种工具：`stack()` 和 `melt()`。


## 3.1 `stack()`：把列名"叠"进行索引

`stack()` 的含义是：**把列名搬下来，和原来的行索引组成一个多级索引**。


In [ ]:
print('原始宽表：')
print(wide)

stacked = wide.stack()
print('\nstack() 之后 —— 变成一个 Series，索引是两级（姓名 + 科目）：')
print(stacked)


In [ ]:
print('类型：', type(stacked).__name__)
print('索引层级名：', stacked.index.names)
print('Series 的名字：', stacked.name, '（None 表示还没名字）')


> **原稿的问题**：原稿写了这么一行来给结果命名 ——
>
> ```python
> stacked_series.columns = ['subject']
> ```
>
> 这行**不会报错，但也完全没生效**。因为它是 `Series`，`Series` 根本没有 `columns`
> 这个属性（只有 `DataFrame` 才有列）。在 pandas 3.0 里给 Series 硬设 `.columns`
> 会**静默地**挂一个没用的属性上去 —— 既不报错、也不影响数据，属于最难查的那种"假成功"。
>
> **正确写法**是 `rename`（改名字）或直接给 `.name` 赋值：


In [ ]:
# 【正确姿势 1】rename 给 Series 起名
print(stacked.rename('成绩').head(3))


In [ ]:
# 【正确姿势 2】直接改 .name
stacked2 = stacked.copy()
stacked2.name = '成绩'
print(stacked2.head(3))


In [ ]:
# 【常用姿势 3】stack 后转成"干净的两列长表"
res = stacked2.rename_axis(index=['姓名', '科目']).reset_index(name='成绩')
print('stack 之后 reset_index，得到标准长表：')
print(res)


## 3.2 `melt()`：更可控的宽变长

`stack()` 会**把全部列**都叠下来，有时候你只想叠一部分。
`melt()` 更灵活：可以指定**哪些列不动**（`id_vars`），只叠剩下的。

```python
pd.melt(
    frame,                # 要转换的 DataFrame
    id_vars=None,         # 保持不变的"标识列"（不参与宽转长）
    value_vars=None,      # 要转换的"值列"；默认 = 除 id_vars 外的所有列
    var_name=None,        # 存放"原列名"的新列名（默认 variable）
    value_name='value',   # 存放"原列值"的新列名（默认 value）
    col_level=None,       # 多层列索引时，指定要操作的层级
    ignore_index=True     # 是否忽略原索引，重新生成 0,1,2... （默认 True）
)
```


In [ ]:
df = pd.DataFrame({
    '姓名': ['张三', '李四', '王五'],
    '数学': [90, 88, 95],
    '英语': [85, 92, 89],
    '语文': [88, 90, 93],
})
print('原始宽表：')
print(df)


In [ ]:
df_melt = pd.melt(df, id_vars='姓名', var_name='科目', value_name='成绩')
print('melt 之后（姓名不动，其余三列叠成长表）：')
print(df_melt)


In [ ]:
# 对比：id_vars 传多个，就有多列"不动"
df2 = df.copy()
df2.insert(1, '班级', ['一班', '一班', '二班'])

print('多了一个"班级"列：')
print(df2)

print('\nid_vars=["姓名","班级"] —— 这两列都保持不变：')
print(pd.melt(df2, id_vars=['姓名', '班级'], var_name='科目', value_name='成绩'))


> **`stack` 和 `melt` 怎么选？**
>
> | 场景 | 用哪个 |
> |---|---|
> | 所有列都要叠下去 | `stack()`（更省事） |
> | 只叠部分列、要保留标识列 | `melt()`（更常用） |
> | 结果想要 Series 的多级索引 | `stack()` |
> | 结果想要三列的扁平表（最常用） | `melt()` |


---
# 四、长变宽（行转列）

反方向：把长表变回宽表。三个工具 `unstack` / `pivot` / `pivot_table`。


## 4.1 `unstack()`：`stack` 的逆操作

先把长表变成"多级索引 + 一列数值"的形状，然后 `unstack()` 把某一级索引抬回列名。


In [ ]:
# 准备长表（复用上面的 df_melt）
df_melt = df_melt.copy()
print('长表原始样子：')
print(df_melt)

# 把 姓名 + 科目 设成多级索引，成绩留作普通数值列
long_idx = df_melt.set_index(['姓名', '科目'])
print('\n设成多级索引后：')
print(long_idx)


In [ ]:
# unstack() 默认把【最内层】索引（这里是"科目"）抬成列名
unstacked = long_idx.unstack()
print('unstack() 之后（列变成两层）：')
print(unstacked)

print('\n列索引的层级：')
print('第 0 层 =', list(unstacked.columns.get_level_values(0)))
print('第 1 层 =', list(unstacked.columns.get_level_values(1)))
print('列索引名称 =', unstacked.columns.name)


In [ ]:
# 上面第 0 层全是"成绩"（也就是数值列的名字），是多余的，用 droplevel 丢掉
unstacked.columns = unstacked.columns.droplevel(0)
unstacked.columns.name = None      # 顺手清掉列索引的名字，更干净
print('丢掉多余的一层后：')
print(unstacked)


In [ ]:
# 最后把"姓名"从索引变回普通列，就和最开始的宽表一模一样了
restored = unstacked.reset_index()
print('最终还原的宽表：')
print(restored)


> **原稿的问题**：原稿最后一句写的是
>
> ```python
> df_melt_index.reset_index()
> ```
>
> **没有把返回值赋给任何变量**。而 `reset_index()` **不改原对象、只返回新对象**，
> 所以这行执行完，结果直接被丢掉了 —— 你以为还原了，其实什么都没发生。
>
> 这也是 pandas 里最经典的坑之一：**凡是"返回新对象"的方法，都要接住返回值。**


## 4.2 `pivot()`：一步到位做透视

上面 `set_index → unstack → droplevel → reset_index` 四步太啰嗦，
`pivot()` 一句话就能干完：**指定谁当行（index）、谁当列（columns）、谁当值（values）**。


In [ ]:
df_pivot = df_melt.pivot(index='姓名', columns='科目', values='成绩')
print('pivot 一步得到宽表：')
print(df_pivot)

print('\n列索引名 =', df_pivot.columns.name, '，行索引名 =', df_pivot.index.name)


In [ ]:
# 通常最后会把索引和列名"清干净"，让它跟普通表一样
clean = df_pivot.reset_index()
clean.columns.name = None
print('清理后的宽表：')
print(clean)


## 4.3 `pivot_table()`：`pivot()` 的加强版

`pivot()` 有个硬限制：**行 + 列的组合必须唯一**，否则报错。
现实中经常有重复，这时候就该 `pivot_table()` —— 它能顺便做**聚合**（默认求平均）。


In [ ]:
# 造一份"有重复"的数据：小红在同一天考了两次数学
dup = pd.DataFrame({
    '姓名': ['小明', '小明', '小红', '小红'],
    '科目': ['数学', '语文', '数学', '数学'],
    '成绩': [90, 88, 85, 95],     # 小红的两条数学 85 和 95
})
print('有重复的数据：')
print(dup)

print('\npivot() 直接报错（组合不唯一）：')
try:
    dup.pivot(index='姓名', columns='科目', values='成绩')
except ValueError as e:
    print('  ValueError:', str(e)[:100])


In [ ]:
print('pivot_table() 可以，默认 aggfunc="mean" 求平均：')
print(dup.pivot_table(index='姓名', columns='科目', values='成绩', aggfunc='mean'))

print('\n换成求和 aggfunc="sum"：')
print(dup.pivot_table(index='姓名', columns='科目', values='成绩', aggfunc='sum', fill_value=0))


> **`pivot` vs `pivot_table` 速查**
>
> | | `pivot()` | `pivot_table()` |
> |---|---|---|
> | 组合不唯一时 | ❌ 报错 | ✅ 自动聚合 |
> | 默认行为 | 直接重排 | 按 `mean` 聚合 |
> | 缺失处理 | 无 | `fill_value` 参数可填 |
> | 适合场景 | 数据已经"扁平且唯一" | **真实脏数据**（推荐） |


---
# 五、DataFrame ↔ 字典：`to_dict()`

和外部系统（接口、JSON）打交道时，经常要把 DataFrame 转成字典。
`to_dict()` 的重点是 **`orient` 参数** —— 它决定"字典的键用什么"。


In [ ]:
d = pd.DataFrame({
    '姓名': ['张三', '李四'],
    '年龄': [25, 30],
}, index=['a', 'b'])

print('原始 DataFrame：')
print(d)


In [ ]:
print("orient='dict'（默认）：键 = 列名，值 = {索引: 值}")
print(d.to_dict(orient='dict'))

print("\norient='list'：键 = 列名，值 = 值的列表")
print(d.to_dict(orient='list'))

print("\norient='index'：键 = 行索引，值 = {列名: 值}  <- 转 JSON 最常用")
print(d.to_dict(orient='index'))


In [ ]:
print("orient='records'：每行一个字典组成的列表  <- 接口交互最常用")
print(d.to_dict(orient='records'))

print("\norient='split'：把 index / columns / data 拆成三部分")
print(d.to_dict(orient='split'))

print("\norient='tight'：在 split 基础上再带上各种名字")
print(d.to_dict(orient='tight'))


**`orient` 对照表（7 种，按常用度排）：**

| `orient` | 结构 | 典型用途 |
|---|---|---|
| `'records'` | `[{列:值}, {列:值}]` | **对接口收发 JSON** |
| `'index'` | `{索引: {列:值}}` | 按主键查数据 |
| `'dict'`（默认） | `{列: {索引: 值}}` | 按列取一列 |
| `'list'` | `{列: [值, 值]}` | 转成普通 dict 再加工 |
| `'series'` | `{列: Series}` | 想保留 Series 类型时 |
| `'split'` | `{index, columns, data}` | 保留完整结构信息 |
| `'tight'` | split + 各种名称 | 需要无损往返时 |

反方向用 `pd.DataFrame(dict)` 或 `pd.DataFrame.from_records(...)` 就能建回表。


In [ ]:
# 反方向：由 records 建回 DataFrame
rows = d.to_dict(orient='records')
print('records：', rows)
print('\n用 DataFrame(rows) 建回表：')
print(pd.DataFrame(rows))


---
# 常见坑清单

| 坑 | 现象 | 正解 |
|---|---|---|
| `Series.columns = [...]` | **静默无效**，不报错也不生效 | Series 用 `.rename()` / `.name` |
| `reset_index()` 不赋值 | 结果被丢弃 | 一定要接住返回值 |
| `str.split()` 忘了 `expand=True` | 得到一列列表，不是多列 | 加 `expand=True` |
| 拆分后列名写死 | 列数一变就报错 | 用 `add_prefix()` 或 `join()` |
| 该 `explode` 却用 `split` | 形状不对（要长不要宽） | 一列列表炸多行用 `explode` |
| 重复组合上硬用 `pivot()` | `ValueError: Index contains duplicate entries` | 改用 `pivot_table()` |
| `unstack()` 后列是两层 | 列名看着别扭 | `droplevel(0)` + `columns.name = None` |
| `melt` 后列名叫 `variable`/`value` | 看不懂 | 指定 `var_name` / `value_name` |


---
# 动手练习

1. **拆地址**：给定 `'浙江省-杭州市-西湖区'` 这类字符串组成的 `Series`，
   拆成「省 / 市 / 区」三列。思考：如果有的地址只有两级，会怎样？
2. **宽变长**：造一张 3 个学生 × 4 门课的宽表，用 `melt` 变成长表，
   再用 `pivot` 变回去，验证两次结果一致。
3. **长变宽 + 聚合**：造一份"日期 + 商品 + 销量"的流水，
   用 `pivot_table` 得到「行=日期、列=商品、值=销量合计」的宽表。
4. **`to_dict` 选择**：一个 DataFrame 想传给前端渲染表格，
   应该用哪个 `orient`？想按用户 ID 快速查一条记录呢？
5. **动手找 bug**：下面这段代码想给 stack 后的结果起名，为什么不生效？怎么改？

   ```python
   s = df.stack()
   s.columns = ['成绩']
   ```


---
# 答案思路

**1. 拆地址**

```python
s = pd.Series(['浙江省-杭州市-西湖区', '广东省-深圳市', '北京市-朝阳区'])
parts = s.str.split('-', expand=True)
parts.columns = ['省', '市', '区']
print(parts)
# 注意第 2 行只有两级（北京市 / 朝阳区），会自动补 NaN —— "市"列是空的
# 这类数据最好先统一格式，否则列的含义会错位
```

**2. 宽变长再变回**

```python
w = pd.DataFrame({'数学': [1,2,3], '语文': [4,5,6], '英语': [7,8,9], '物理': [10,11,12]},
                 index=['A','B','C'])
l = w.melt(ignore_index=False, var_name='科目', value_name='分数')
back = l.pivot(index=l.index, columns='科目', values='分数')
back.columns.name = None
print((back[w.columns] == w).all().all())   # True，两次转换无损
```

**3. 长变宽 + 聚合**

```python
flow = pd.DataFrame({
    '日期': ['2024-01-01','2024-01-01','2024-01-02','2024-01-02','2024-01-02'],
    '商品': ['A','B','A','A','B'],
    '销量': [1,2,3,4,5],
})
pt = flow.pivot_table(index='日期', columns='商品', values='销量', aggfunc='sum', fill_value=0)
pt.columns.name = None
print(pt.reset_index())
```

**4. `to_dict` 选择**

```python
# 给前端渲染表格 -> 每行一条记录，用 records
payload = df.to_dict(orient='records')     # [{'姓名':'张三',...}, ...]

# 按用户 ID 快速查 -> 键就是索引，用 index
lookup = df.set_index('user_id').to_dict(orient='index')   # {101: {...}, 102: {...}}
```

**5. 找 bug**

```python
s = df.stack()
s.columns = ['成绩']      # ❌ Series 没有 columns，这行静默无效

s = df.stack().rename('成绩')   # ✅ 正确：用 rename 改 Series 的名字
# 或者： s.name = '成绩'
```
